# **01. Project Landing and Exploratory Data Analysis**

This notebook starts the customer support AI implementation using the multilingual Kaggle support-ticket dataset. Kaggle provides five CSV files, so we first audit all five files and then use only the compatible files for modelling.

In this notebook we check:

- dataset size and column meanings for each CSV
- missing values and duplicate rows
- queue and priority label balance
- language and ticket-type distributions
- text length and text variety
- columns that should not be used for initial prediction because they may cause target leakage
- which CSV files are suitable to merge for the final model

# **1. Environment Setup**

We import project helpers and locate the repository root. This keeps the notebook portable for every group member who places the Kaggle CSV files in `data/raw/`.

In [1]:
# Import libraries and project helper functions.
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'customer_support_ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from customer_support_ai.config import (
    CATEGORY_COLUMN_CANDIDATES,
    AUDIT_ONLY_DATASET_FILENAMES,
    PRIORITY_COLUMN_CANDIDATES,
    COMPATIBLE_DATASET_FILENAMES,
    COMPATIBLE_DATASET_PATHS,
    DATA_RAW_DIR,
)
from customer_support_ai.data_preprocessing import find_column, identify_leakage_columns, load_project_dataset, normalise_columns

# **2. Audit Kaggle CSV Files**

The Kaggle files are kept local and are not committed to GitHub. We inspect all five CSVs first, then choose the files that share the same prediction targets.

In [2]:
# Check that all Kaggle CSV files are available locally.
all_dataset_filenames = COMPATIBLE_DATASET_FILENAMES + AUDIT_ONLY_DATASET_FILENAMES
all_dataset_paths = [DATA_RAW_DIR / filename for filename in all_dataset_filenames]
missing_files = [path.name for path in all_dataset_paths if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        'Place these Kaggle CSV files in data/raw/: ' + ', '.join(missing_files)
    )

audit_rows = []
for path in all_dataset_paths:
    temp_df = normalise_columns(pd.read_csv(path))
    audit_rows.append({
        'file': path.name,
        'rows': len(temp_df),
        'columns': temp_df.shape[1],
        'languages': ', '.join(sorted(temp_df['language'].dropna().astype(str).unique())) if 'language' in temp_df.columns else 'missing',
        'queue_labels': temp_df['queue'].nunique(dropna=True) if 'queue' in temp_df.columns else 0,
        'priority_labels': temp_df['priority'].nunique(dropna=True) if 'priority' in temp_df.columns else 0,
        'compatible_for_main_model': path.name in COMPATIBLE_DATASET_FILENAMES,
    })

dataset_audit = pd.DataFrame(audit_rows)
dataset_audit

,file,rows,columns,languages,queue_labels,priority_labels,compatible_for_main_model
0,aa_dataset-tickets-multi-lang-5-2-50-version.csv,28587,16,"de, en",10,3,True
1,dataset-tickets-multi-lang-4-20k.csv,20000,15,"de, en",10,3,True
2,dataset-tickets-multi-lang3-4k.csv,4000,17,"de, en, es, fr, pt",10,3,True
3,dataset-tickets-german_normalized.csv,2125,5,de,10,3,False
4,dataset-tickets-german_normalized_50_5_2.csv,13178,5,de,42,5,False


# **3. Load Compatible Modelling Dataset**

The three compatible CSV files use the same `queue` and `priority` label design. The two German-normalized files are useful for discussion, but their queue taxonomies are different, so we do not merge them into the main supervised classifier.

In [3]:
# Load the compatible CSVs and remove duplicate tickets by subject + body.
raw_df = load_project_dataset(COMPATIBLE_DATASET_PATHS)
df = raw_df.copy()
category_col = find_column(df, CATEGORY_COLUMN_CANDIDATES)
priority_col = find_column(df, PRIORITY_COLUMN_CANDIDATES)

print('Compatible CSV files:')
for filename in raw_df.attrs['dataset_files']:
    print(f'- {filename}')
print(f"Raw rows before deduplication: {raw_df.attrs['raw_rows_before_dedup']:,}")
print(f"Duplicate rows removed: {raw_df.attrs['duplicate_rows_removed']:,}")
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.head()

Compatible CSV files:
- aa_dataset-tickets-multi-lang-5-2-50-version.csv
- dataset-tickets-multi-lang-4-20k.csv
- dataset-tickets-multi-lang3-4k.csv
Raw rows before deduplication: 52,587
Duplicate rows removed: 8,309
Rows: 44,278
Columns: 19


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,source_file,business_type,tag_9
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN


# **3. Dataset Structure**

This compact data dictionary helps us explain which fields are useful for modelling and which fields are only useful for analysis.

In [4]:
# Summarise column names, data types, and example values.
data_dictionary = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[col].dtype) for col in df.columns],
    'non_null_rows': [df[col].notna().sum() for col in df.columns],
    'unique_values': [df[col].nunique(dropna=True) for col in df.columns],
    'example_value': [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
})

data_dictionary

,column,dtype,non_null_rows,unique_values,example_value
0,subject,str,39690,36971,Wesentlicher Sicherheitsvorfall
1,body,str,44275,44183,"Sehr geehrtes Support-Team,\n\nich möchte eine..."
2,answer,str,44269,44079,Vielen Dank für die Meldung des kritischen Sic...
3,type,str,44278,4,Incident
4,queue,str,44278,10,Technical Support
5,priority,str,44278,3,high
6,language,str,44278,5,de
7,version,float64,28586,3,51.0
8,tag_1,str,44278,251,Security
9,tag_2,str,44219,405,Outage


# **4. Data Quality Checks**

Before modelling, we check duplicate records and missing values. Missing tags are expected because not every ticket has eight tags.

In [5]:
# Check duplicate rows and missing values.
quality_summary = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'duplicate_rows': df.duplicated().sum(),
    'duplicate_rate': df.duplicated().mean(),
})

quality_summary

rows              44278.0
columns              19.0
duplicate_rows        0.0
duplicate_rate        0.0
dtype: float64

In [6]:
# Show only columns with missing values.
missing_summary = (
    df.isna()
    .sum()
    .rename('missing_rows')
    .to_frame()
    .assign(missing_rate=lambda table: table['missing_rows'] / len(df))
    .query('missing_rows > 0')
    .sort_values('missing_rows', ascending=False)
)

missing_summary

,missing_rows,missing_rate
tag_9,44278,1.000000
tag_8,41553,0.938457
business_type,40278,0.909662
tag_7,37404,0.844754
tag_6,29807,0.673179
tag_5,16760,0.378518
version,15692,0.354397
subject,4588,0.103618
tag_4,3519,0.079475
tag_3,198,0.004472


# **5. Target Label Distributions**

The main prediction targets are `queue` for routing/category prediction and `priority` for urgency prediction.

In [7]:
# Check queue balance.
queue_distribution = (
    df[category_col]
    .value_counts(dropna=False)
    .rename_axis('queue')
    .reset_index(name='count')
)
queue_distribution['share'] = queue_distribution['count'] / len(df)
queue_distribution

,queue,count,share
0,Technical Support,13095,0.295745
1,Product Support,8113,0.183229
2,Customer Service,6741,0.152243
3,IT Support,5237,0.118275
4,Billing and Payments,4368,0.098649
5,Returns and Exchanges,2204,0.049776
6,Service Outages and Maintenance,1710,0.038620
7,Sales and Pre-Sales,1393,0.031460
8,Human Resources,802,0.018113
9,General Inquiry,615,0.013890


In [8]:
# Check priority balance.
priority_distribution = (
    df[priority_col]
    .value_counts(dropna=False)
    .rename_axis('priority')
    .reset_index(name='count')
)
priority_distribution['share'] = priority_distribution['count'] / len(df)
priority_distribution

,priority,count,share
0,medium,17961,0.405642
1,high,17355,0.391955
2,low,8962,0.202403


# **6. Operational and Language Distributions**

Language, ticket type, and tags help us understand the dataset and can support feature engineering or responsible AI discussion.

In [9]:
# Review key operational fields.
operational_columns = ['language', 'type', 'tag_1', 'tag_2', 'tag_3', 'tag_4']

for col in operational_columns:
    if col in df.columns:
        print()
        print(col)
        display(df[col].value_counts(dropna=False).head(12).to_frame('count'))


language


,count
language,
en,25137
de,17380
es,812
fr,476
pt,473



type


,count
type,
Incident,17758
Request,12714
Problem,9271
Change,4535



tag_1


...

,count
tag_2,
Performance,6578
IT,3016
Bug,2705
Documentation,2309
Disruption,1995
Product,1858
Sales,1808
Outage,1772
Feedback,1724



tag_3


,count
tag_3,
IT,6903
Performance,3429
Tech Support,2950
Documentation,2311
Feedback,2204
Disruption,2025
Feature,1789
Bug,1646
Outage,1480



tag_4


,count
tag_4,
Tech Support,7331
IT,5343
NaN,3519
Documentation,2542
Performance,1617
Feedback,1520
Disruption,1347
Recovery,1226
Resolution,1128


# **7. Text Length and Variety Analysis**

This dataset has unique ticket bodies, which gives the NLP model a better chance to learn useful text patterns from the customer message.

In [10]:
# Combine subject and body for text analysis.
df['subject_clean'] = df['subject'].fillna('').astype(str)
df['ticket_text'] = (df['subject_clean'] + ' ' + df['body'].fillna('').astype(str)).str.strip()
df['ticket_word_count'] = df['ticket_text'].str.split().str.len()
df['ticket_char_count'] = df['ticket_text'].str.len()

text_variety = pd.Series({
    'total_tickets': len(df),
    'unique_subjects': df['subject'].nunique(dropna=True),
    'unique_bodies': df['body'].nunique(dropna=True),
    'unique_ticket_text_values': df['ticket_text'].nunique(dropna=True),
    'ticket_text_uniqueness_rate': df['ticket_text'].nunique(dropna=True) / len(df),
})

text_variety

total_tickets                  44278.0
unique_subjects                36971.0
unique_bodies                  44183.0
unique_ticket_text_values      44278.0
ticket_text_uniqueness_rate        1.0
dtype: float64

In [11]:
# Summarise ticket text length.
text_length_summary = df[['ticket_word_count', 'ticket_char_count']].describe().T
text_length_summary

,count,mean,std,min,25%,50%,75%,max
ticket_word_count,44278.0,65.728917,40.856920,1.0,36.0,61.0,87.0,394.0
ticket_char_count,44278.0,470.116085,283.558894,4.0,264.0,439.0,614.0,2923.0


In [12]:
# Compare average text length by queue and priority.
length_by_queue = (
    df.groupby(category_col)['ticket_word_count']
    .agg(['count', 'mean', 'median'])
    .sort_values('mean', ascending=False)
)

length_by_priority = (
    df.groupby(priority_col)['ticket_word_count']
    .agg(['count', 'mean', 'median'])
    .sort_values('mean', ascending=False)
)

print('Average word count by queue')
display(length_by_queue)
print('Average word count by priority')
display(length_by_priority)

Average word count by queue


,count,mean,median
queue,,,
General Inquiry,615,67.032520,69.0
Technical Support,13095,66.867201,62.0
Service Outages and Maintenance,1710,66.857895,63.0
Billing and Payments,4368,66.617674,63.0
Customer Service,6741,66.213025,62.0
Product Support,8113,65.005300,61.0
IT Support,5237,64.407485,60.0
Sales and Pre-Sales,1393,63.687724,61.0
Returns and Exchanges,2204,63.125681,59.0


Average word count by priority


,count,mean,median
priority,,,
high,17355,66.002247,61.0
medium,17961,65.797728,62.0
low,8962,65.061705,61.0


# **8. Target Leakage Review**

The `answer` field is useful for understanding responses, but it should not be used for initial queue or priority prediction because it is created after the customer request is handled.

In [13]:
# Identify columns excluded from initial prediction features.
leakage_columns = identify_leakage_columns(df)

pd.DataFrame({
    'excluded_column': leakage_columns,
    'reason': 'Potentially known after ticket handling or directly related to response outcome',
})

,excluded_column,reason
0,answer,Potentially known after ticket handling or dir...


# **9. EDA Findings for Report**

This final cell summarises the main EDA findings in report-ready language.

In [14]:
# Generate a short findings summary from the EDA results.
findings = [
    f'The dataset contains {len(df):,} tickets and {raw_df.shape[1]} original columns.',
    f'There are {df.duplicated().sum():,} duplicate rows.',
    f'The queue target has {df[category_col].nunique()} classes and is moderately imbalanced.',
    f'The priority target has {df[priority_col].nunique()} classes: {", ".join(priority_distribution["priority"].astype(str))}.',
    f'The dataset is multilingual, with {df["language"].nunique()} languages represented.',
    f'The ticket body field has {df["body"].nunique():,} unique values across {len(df):,} tickets.',
    f'The average combined subject/body length is {df["ticket_word_count"].mean():.1f} words.',
    f'Potential leakage columns excluded from modelling: {", ".join(leakage_columns)}.',
]

for item in findings:
    print('-', item)

- The dataset contains 44,278 tickets and 19 original columns.
- There are 0 duplicate rows.
- The queue target has 10 classes and is moderately imbalanced.
- The priority target has 3 classes: medium, high, low.
- The dataset is multilingual, with 5 languages represented.
- The ticket body field has 44,183 unique values across 44,278 tickets.
- The average combined subject/body length is 65.7 words.
- Potential leakage columns excluded from modelling: answer.
